<a href="https://colab.research.google.com/github/summiyazeb/WEB-SEARCH-ENGINE/blob/main/Notebooks/02-Lexicon-and-Indexes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
#
# Cell 1A: mount Google Drive and define file paths
#
import os
import json
import time
from google.colab import drive

# Connect Colab to Google Drive
drive.mount('/content/drive')

# Folder paths
BASE_DIR = '/content/drive/MyDrive/DSA_Search_Engine' #the project folder.
INDEX_DIR = os.path.join(BASE_DIR, 'index_data')   # all index outputs go here

# --- Input / output file paths ---
DATASET_PATH = os.path.join(BASE_DIR, 'cleaned_dataset.jsonl')   # full 55,000 reviews
SAMPLE_PATH  = os.path.join(BASE_DIR, 'sample_dataset.jsonl')    # 1,000-review test file

# Create index_data/ if it doesn't exist yet
os.makedirs(INDEX_DIR, exist_ok=True)

#  Quick check that everything is where we expect
print("Dataset found :", os.path.exists(DATASET_PATH))
print("index_data/   :", os.path.exists(INDEX_DIR))

Mounted at /content/drive
Dataset found : True
index_data/   : True


In [6]:
#
# Cell 1B: Tokenizer : cuts a piece of text into a clean list of words that the indexes can use.
# LOWERCASE each word
# trims extra appostrophies
#Stemming: coverts words to rooot words, eg drifting to drift

import re # regex: we use ot to find words
from functools import lru_cache # its a memoery, once we have found smth, it will remeber it, reduces time
import nltk # human lang library for stemming and stopwords
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer # rules for stemming, words-> roots

nltk.download('stopwords', quiet=True) # to download it and quiet if ale exists

STOPWORDS = set(stopwords.words('english')) # returns list of eng stopwords
_stemmer = PorterStemmer()

@lru_cache(maxsize=None)          # each unique word is stemmed only once
def stem_word(word):              # helps lru_cache to remember words
    return _stemmer.stem(word)

WORD_PATTERN = re.compile(r"[a-z0-9']+")   # a run of letters/digits/apostrophes

def tokenize(text, remove_stopwords=False):
    """Return a list of stemmed lowercase tokens.
    Stopwords are KEPT by default so 'data structures and algorithms'
    can match on 'and' (spec Requirement 5)."""
    tokens = []
    for w in WORD_PATTERN.findall(text.lower()):
        w = w.strip("'")
        if not w:
            continue
        if remove_stopwords and w in STOPWORDS:
            continue
        tokens.append(stem_word(w))
    return tokens

print(tokenize("The battery life is GREAT, but the controller drifts!"))

['the', 'batteri', 'life', 'is', 'great', 'but', 'the', 'control', 'drift']


In [7]:
#
# Cell 1C: Create a 1,000-review sample file for testing
#
SAMPLE_SIZE = 1000

count = 0
# Read the big file one line at a time; write only the first 1,000 lines
# r means read, utf-8 is for handling special characters
# w means write
with open(DATASET_PATH, 'r', encoding='utf-8') as src, \
     open(SAMPLE_PATH, 'w', encoding='utf-8') as dst:

    for line in src:  # to write line by line from src to dst
        dst.write(line)
        count += 1
        if count >= SAMPLE_SIZE:
            break

print(f"Sample created with {count} reviews") # count no of reviews stored
print(f"Saved at: {SAMPLE_PATH}")

#  Check: read it back, count the lines, inspect one timestamp ---
with open(SAMPLE_PATH, 'r', encoding='utf-8') as f:
    records = [json.loads(line) for line in f]

print(f"Lines in sample file: {len(records)}")
print(f"First doc_id: {records[0]['doc_id']} | Last doc_id: {records[-1]['doc_id']}")
print(f"Example timestamp: {records[0]['timestamp']}")

Sample created with 1000 reviews
Saved at: /content/drive/MyDrive/DSA_Search_Engine/sample_dataset.jsonl
Lines in sample file: 1000
First doc_id: 0 | Last doc_id: 999
Example timestamp: 1602718512453
